NaN analizė

In [23]:
import pandas

from config import features, unscaled_csv, pollutants

unscaled_df = pandas.read_csv(unscaled_csv, parse_dates=['date'])

total = len(unscaled_df)
nan_rows = unscaled_df[features].isnull().any(axis=1).sum()
nan_cells = unscaled_df[features].isnull().sum().sum()
print(f"Eilutės su bent vienu NaN: {nan_rows} iš {total} ({nan_rows / total * 100:.1f}%)")
print(f"Iš viso NaN reikšmių: {nan_cells}")
print(f"Vidutiniškai NaN stulpelių per NaN eilutę: {nan_cells / nan_rows:.1f}")

Eilutės su bent vienu NaN: 13855 iš 96211 (14.4%)
Iš viso NaN reikšmių: 29606
Vidutiniškai NaN stulpelių per NaN eilutę: 2.1


NaN per stulpelį

In [24]:
print(unscaled_df.isnull().sum())

print(len(unscaled_df))
print((unscaled_df.isnull().sum() / len(unscaled_df)) * 100)

date               0
time               0
O3              2922
NO              2748
NO2             2748
PM10            5651
PM2.5           5221
wind_speed      2579
temp            2579
day_category       0
wind_dir_sin    2579
wind_dir_cos    2579
dtype: int64
96211
date            0.000000
time            0.000000
O3              3.037075
NO              2.856222
NO2             2.856222
PM10            5.873549
PM2.5           5.426614
wind_speed      2.680567
temp            2.680567
day_category    0.000000
wind_dir_sin    2.680567
wind_dir_cos    2.680567
dtype: float64


Teršalų koreliacija

In [25]:
# print(unscaled_df[pollutants].corr()['CO'].round(2))

Tarpų analizė

In [26]:
for col in features:
    is_null = unscaled_df[col].isnull()
    groups = is_null.ne(is_null.shift()).cumsum()
    gap_sizes = unscaled_df[is_null].groupby(groups).size()

    print(f"\n=== {col} — tarpų: {len(gap_sizes)}, iš viso NaN: {is_null.sum()}, ilgiausias: {gap_sizes.max() if len(gap_sizes) > 0 else 0} val. ===")
    print(f"  1 val.:     {(gap_sizes == 1).sum()}")
    print(f"  2-5 val.:   {((gap_sizes >= 2) & (gap_sizes <= 5)).sum()}")
    print(f"  6-12 val.:  {((gap_sizes >= 6) & (gap_sizes <= 12)).sum()}")
    print(f"  12-24 val.: {((gap_sizes >= 13) & (gap_sizes <= 24)).sum()}")
    print(f"  24+ val.:   {(gap_sizes > 24).sum()}")


=== O3 — tarpų: 468, iš viso NaN: 2922, ilgiausias: 219 val. ===
  1 val.:     265
  2-5 val.:   110
  6-12 val.:  33
  12-24 val.: 34
  24+ val.:   26

=== NO — tarpų: 419, iš viso NaN: 2748, ilgiausias: 314 val. ===
  1 val.:     251
  2-5 val.:   111
  6-12 val.:  16
  12-24 val.: 12
  24+ val.:   29

=== NO2 — tarpų: 419, iš viso NaN: 2748, ilgiausias: 314 val. ===
  1 val.:     251
  2-5 val.:   111
  6-12 val.:  16
  12-24 val.: 12
  24+ val.:   29

=== PM10 — tarpų: 400, iš viso NaN: 5651, ilgiausias: 719 val. ===
  1 val.:     190
  2-5 val.:   108
  6-12 val.:  21
  12-24 val.: 14
  24+ val.:   67

=== PM2.5 — tarpų: 480, iš viso NaN: 5221, ilgiausias: 344 val. ===
  1 val.:     239
  2-5 val.:   139
  6-12 val.:  26
  12-24 val.: 17
  24+ val.:   59

=== wind_speed — tarpų: 82, iš viso NaN: 2579, ilgiausias: 144 val. ===
  1 val.:     0
  2-5 val.:   0
  6-12 val.:  0
  12-24 val.: 73
  24+ val.:   9

=== temp — tarpų: 82, iš viso NaN: 2579, ilgiausias: 144 val. ===
  1 val.

Tarpai per metus — gal kurie metai ypač prasti?

In [27]:
unscaled_df['year'] = unscaled_df['date'].dt.year
print(unscaled_df.groupby('year')[['PM10', 'PM2.5', 'O3', 'NO', 'NO2']].apply(lambda x: x.isnull().sum()))

      PM10  PM2.5   O3   NO  NO2
year                            
2012  1296    694  344  498  498
2013   491    276  142  120  120
2014   605    275  400  112  112
2015   258    262  159  120  120
2016   404    349  246  121  121
2017   315    331  155   89   89
2018   300    913  129  140  140
2019   314    812  186  474  474
2023   507    520  450  406  406
2024   713    473  179  430  430
2025   448    316  532  238  238


Išmetus CO

In [28]:
features_no_co = [f for f in features if f != 'CO']


nan_without_co = unscaled_df[features_no_co].isnull().any(axis=1).sum()
print(f"Be CO:  {nan_without_co} iš {len(unscaled_df)} ({nan_without_co / len(unscaled_df) * 100:.1f}%)")

Be CO:  13855 iš 96211 (14.4%)


Interpoliavimo nauda — kiek NaN reikšmių ir pilnų eilučių atgautume

In [29]:
def get_gap_labels(series):
    """Kiekvienai NaN reikšmei priskiria to tarpo ilgį."""
    is_null = series.isnull()
    groups = is_null.ne(is_null.shift()).cumsum()
    gap_sizes = series[is_null].groupby(groups).transform('size')
    result = pandas.Series(0, index=series.index)
    result[is_null] = gap_sizes
    return result

gap_label_df = pandas.DataFrame({
    col: get_gap_labels(unscaled_df[col])
    for col in features_no_co
})

total_rows = len(unscaled_df)

max_gap_per_row = gap_label_df.max(axis=1)
nan_rows = max_gap_per_row[max_gap_per_row > 0]
total_nan_rows = len(nan_rows)

print(f"{'Tarpo ilgis':<14} {'Eilutės':>14} {'% visų NaN':>12} {'Kaupiamasis %':>14}")
print("-" * 58)
cumulative = 0
categories = [(1,1,'1 val.'),(2,2,'2 val.'),(3,5,'3–5 val.'),(6,12,'6–12 val.'),(13,24,'12–24 val.'),(25,999999,'24+ val.')]
for lo, hi, lbl in categories:
    n = ((nan_rows >= lo) & (nan_rows <= hi)).sum()
    pct = n / total_nan_rows * 100
    cumulative += pct
    print(f"{lbl:<14} {n:>14,} {pct:>11.1f}% {cumulative:>13.1f}%")
print(f"{'Iš viso':<14} {total_nan_rows:>14,} {'100.0%':>12}")

print()
print(f"{'Interpoliuojami tarpai':<24} {'Pilnos eilutės':>16} {'% visų':>10} {'Prieaugis':>12}")
print("-" * 64)
baseline = (gap_label_df == 0).all(axis=1).sum()
prev = baseline
print(f"{'Nėra interpoliacijos':<24} {baseline:>16,} {baseline/total_rows*100:>9.1f}%")
for max_gap, lbl in [(1,'≤ 1 val.'),(2,'≤ 2 val.'),(5,'≤ 5 val.'),(12,'≤ 12 val.'),(24,'≤ 24 val.')]:
    remaining_nan = (gap_label_df > 0) & (gap_label_df > max_gap)
    complete = (~remaining_nan.any(axis=1)).sum()
    gain = complete - prev
    print(f"{lbl:<24} {complete:>16,} {complete/total_rows*100:>9.1f}% {gain:>+12,}")
    prev = complete

Tarpo ilgis           Eilutės   % visų NaN  Kaupiamasis %
----------------------------------------------------------
1 val.                    470         3.4%           3.4%
2 val.                    301         2.2%           5.6%
3–5 val.                  322         2.3%           7.9%
6–12 val.                 573         4.1%          12.0%
12–24 val.              2,612        18.9%          30.9%
24+ val.                9,577        69.1%         100.0%
Iš viso                13,855       100.0%

Interpoliuojami tarpai     Pilnos eilutės     % visų    Prieaugis
----------------------------------------------------------------
Nėra interpoliacijos               82,356      85.6%
≤ 1 val.                           82,826      86.1%         +470
≤ 2 val.                           83,127      86.4%         +301
≤ 5 val.                           83,449      86.7%         +322
≤ 12 val.                          84,022      87.3%         +573
≤ 24 val.                          86,634  

Švarūs 5 valandų blokai per dienos valandas ir kategorijas

In [30]:
for start_hour in range(1, 21):
    target_hours = list(range(start_hour, start_hour + 5))
    for cat in [0, 1, 2]:
        cat_name = ['darbo', 'išeig', 'priešš'][cat]
        subset = unscaled_df[
            (unscaled_df['day_category'] == cat) &
            (unscaled_df['time'].isin(target_hours))
            ]
        clean = subset[features_no_co].notna().all(axis=1)
        clean_days = clean.groupby(subset['date']).apply(lambda g: (len(g) == 5) and g.all()).sum()
        if cat == 0:
            print(f"Val. {start_hour:2d}-{start_hour+4:2d} | ", end="")
        print(f"{cat_name}: {clean_days:4d}", end=" | ")
    print()

Val.  1- 5 | darbo: 1800 | išeig: 1011 | priešš:  479 | 
Val.  2- 6 | darbo: 1820 | išeig: 1029 | priešš:  484 | 
Val.  3- 7 | darbo: 1817 | išeig: 1036 | priešš:  487 | 
Val.  4- 8 | darbo: 1815 | išeig: 1028 | priešš:  489 | 
Val.  5- 9 | darbo: 1794 | išeig: 1025 | priešš:  487 | 
Val.  6-10 | darbo: 1760 | išeig: 1022 | priešš:  486 | 
Val.  7-11 | darbo: 1713 | išeig: 1023 | priešš:  480 | 
Val.  8-12 | darbo: 1647 | išeig: 1019 | priešš:  482 | 
Val.  9-13 | darbo: 1582 | išeig: 1021 | priešš:  482 | 
Val. 10-14 | darbo: 1550 | išeig: 1027 | priešš:  479 | 
Val. 11-15 | darbo: 1530 | išeig: 1038 | priešš:  478 | 
Val. 12-16 | darbo: 1558 | išeig: 1035 | priešš:  485 | 
Val. 13-17 | darbo: 1612 | išeig: 1035 | priešš:  491 | 
Val. 14-18 | darbo: 1674 | išeig: 1032 | priešš:  494 | 
Val. 15-19 | darbo: 1739 | išeig: 1030 | priešš:  499 | 
Val. 16-20 | darbo: 1785 | išeig: 1035 | priešš:  504 | 
Val. 17-21 | darbo: 1826 | išeig: 1038 | priešš:  507 | 
Val. 18-22 | darbo: 1847 | išei

Visų features koreliacija

In [31]:
print(unscaled_df[features].corr().round(2))

                O3    NO   NO2  PM10  PM2.5  wind_speed  temp  wind_dir_sin  \
O3            1.00 -0.58 -0.57 -0.37  -0.38        0.21  0.18          0.24   
NO           -0.58  1.00  0.87  0.53   0.48       -0.01  0.00         -0.23   
NO2          -0.57  0.87  1.00  0.50   0.46       -0.02  0.12         -0.17   
PM10         -0.37  0.53  0.50  1.00   0.88       -0.18 -0.10          0.11   
PM2.5        -0.38  0.48  0.46  0.88   1.00       -0.25 -0.12          0.16   
wind_speed    0.21 -0.01 -0.02 -0.18  -0.25        1.00  0.19         -0.19   
temp          0.18  0.00  0.12 -0.10  -0.12        0.19  1.00         -0.08   
wind_dir_sin  0.24 -0.23 -0.17  0.11   0.16       -0.19 -0.08          1.00   
wind_dir_cos  0.28 -0.31 -0.33 -0.20  -0.11       -0.12 -0.16          0.19   

              wind_dir_cos  
O3                    0.28  
NO                   -0.31  
NO2                  -0.33  
PM10                 -0.20  
PM2.5                -0.11  
wind_speed           -0.12  
temp  

Duomenų statistika

In [32]:
print(unscaled_df[features].describe().round(2))


             O3        NO       NO2      PM10     PM2.5  wind_speed      temp  \
count  93289.00  93463.00  93463.00  90560.00  90990.00    93632.00  93632.00   
mean      20.98     98.88     72.05     24.06     15.51        3.46     10.35   
std       17.41    104.49     41.73     13.71     10.92        1.66      6.12   
min        0.00      0.00      0.00      0.00      0.00        0.00    -10.50   
25%        6.59     22.72     40.20     14.60      8.00        2.30      6.00   
50%       16.07     59.43     64.55     21.26     12.90        3.20     10.20   
75%       31.23    140.83     97.60     30.00     19.54        4.40     14.50   
max      151.47    872.83    321.91    187.90    127.60       13.70     33.40   

       wind_dir_sin  wind_dir_cos  
count      93632.00      93632.00  
mean          -0.25         -0.04  
std            0.72          0.65  
min           -1.00         -1.00  
25%           -0.90         -0.63  
50%           -0.53         -0.13  
75%            0.4

Duomenų statistika po normalizavimo

In [34]:
cleaned = unscaled_df.dropna()

cleaned_features = ['O3', 'NO', 'NO2', 'PM10', 'PM2.5', 'wind_speed', 'temp']
print(f"Eilučių po valymo: {len(cleaned)}")
print(cleaned[cleaned_features].describe().round(2))

Eilučių po valymo: 82356
             O3        NO       NO2      PM10     PM2.5  wind_speed      temp
count  82356.00  82356.00  82356.00  82356.00  82356.00    82356.00  82356.00
mean      20.96     97.77     71.66     24.03     15.43        3.46     10.21
std       17.43    103.21     41.43     13.72     10.84        1.66      6.06
min        0.00      0.00      0.00      0.00      0.00        0.00    -10.50
25%        6.54     22.52     40.06     14.60      8.00        2.30      6.00
50%       16.07     59.00     64.20     21.26     12.90        3.20     10.00
75%       31.13    138.68     96.85     30.00     19.40        4.40     14.30
max      151.47    872.83    321.91    187.90    127.60       13.70     33.40


Sezoniniai teršalų trendai pagal mėnesį

In [35]:
unscaled_df['month'] = unscaled_df['date'].dt.month
print(unscaled_df.groupby('month')[['O3', 'NO', 'NO2', 'PM10', 'PM2.5']].mean().round(2))

          O3      NO    NO2   PM10  PM2.5
month                                    
1      15.25  122.16  76.36  26.67  17.08
2      18.50  108.38  72.93  27.38  17.82
3      23.71   92.03  73.72  29.23  19.99
4      30.55   81.20  72.36  25.27  16.63
5      32.42   75.45  68.53  22.69  15.11
6      27.13   87.40  72.36  20.49  13.16
7      22.48   88.32  72.83  19.62  12.91
8      20.33   83.07  68.15  21.05  12.94
9      18.00   95.17  70.47  22.46  14.94
10     14.60  108.68  71.47  23.83  14.58
11     13.09  121.73  71.55  25.44  15.77
12     16.03  122.40  73.90  24.31  15.03
